In [1]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil
from torch.utils.data import DataLoader

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
# You can use any of the 4 train files or all the files for pre-training the legal model



# Training configuration
TARGET_TOKENS = 500_000_000
MAX_SEQ_LEN = 1024  #CONTEXT LENGTH
TRAIN_STEPS = 1
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 512
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
VOCAB_SIZE = 8000
MODEL_DIM = 512
MODEL_LAYERS = 12
MODEL_FF = 4096
N_HEADS = 8
N_KV_HEADS = 8
BATCH_SIZE = 16
# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase2"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

cuda
Configuration loaded.


In [4]:
from huggingface_hub import hf_hub_download

TRAIN_FILES = [
    "Phase_6_Summ/Train/train.json",
]

train_file_paths = {}

for filename in TRAIN_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    train_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

Phase_6_Summ/Train/train.json: reconstructing file:   0%|          |  0.00B /  211MB            

Phase_6_Summ/Train/train.json: downloading bytes:           |  0.00B            

Phase_6_Summ/Train/train.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Train/train.json


In [5]:
from huggingface_hub import hf_hub_download

TEST_FILES = [
    "Phase_6_Summ/Test/test.json",
    
]

test_file_paths = {}

for filename in TEST_FILES:
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=filename,
        repo_type="dataset",
        token=HF_TOKEN
    )

    test_file_paths[filename] = path
    print(f"{filename}")
    print(f"  {path}")

test.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

Phase_6_Summ/Test/test.json
  /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_6_Summ/Test/test.json


In [6]:
import json
import pandas as pd

with open(train_file_paths["Phase_6_Summ/Train/train.json"], "r") as f:
    train_data = json.load(f)

with open(test_file_paths["Phase_6_Summ/Test/test.json"], "r") as f:
    test_data = json.load(f)
    

In [7]:

class Node:
    def __init__(self,value, previous, next, index):
        self.value=value
        self.previous=previous
        self.next=next
        self.index=index


class LinkedArray:
    def __init__(self, items):
        self._array=[None for _ in range(len(items))]
        previous=None
        for i in range(len(items)):
            node=Node(items[i], previous, None, i)
            if previous is not None:
                node.previous=previous
                previous.next=node
            self._array[i]=node
            previous=node

    def get_previous_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].previous == None:
            return None
        return self._array[index].previous.index

    def get_next_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None:
            return None
        return self._array[index].next.index

    def get_by_index(self, index):
        if self._array[index]==None:
            raise ValueError(f"No item at index {index}")
        if self._array[index] == None:
            return None
        return self._array[index].value

    def get_second_next_index(self, index):
        if self._array[index] == None:
            raise ValueError(f"No item at index {index}")
        if self._array[index].next == None or self._array[index].next.next == None:
            return None
        return self._array[index].next.next.index

    def len(self):
        return len(self._array)
    def exists(self, index):
        return (
            0 <= index < len(self._array)
            and self._array[index] is not None
        )

    def replace_pair(self, index, new_item):
        if index > len(self._array)-2 or self._array[index] == None or self._array[index].next == None:
            raise ValueError("Invalid index ",index,new_item)
        self._array[index].value = new_item
        self._array[self._array[index].next.index] = None
        self._array[index].next = self._array[index].next.next
        if self._array[index].next is not None:
            self._array[index].next.previous = self._array[index]


class MaxPriorityMap:
    def __init__(self, heap_key, map_key):
        self._heap_key = heap_key
        self._map_key = map_key
        self._heap = []
        # Map of keys to indexes in the heap
        self._map = {}

    def get_max(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        return self._heap[0]

    def push(self, item):
        self._heap.append(item)
        map_key = self._map_key(item)
        if map_key in self._map:
            raise ValueError("Map key already exist")
        self._map[self._map_key(item)] = len(self._heap) - 1
        self._heapify_up(len(self._heap) - 1)

    def pop(self):
        if len(self._heap) == 0:
            raise ValueError("the heap is empty")
        self._swap(0, len(self._heap) - 1)
        item = self._heap.pop()
        map_key = self._map_key(item)
        self._map.pop(map_key)
        self._heapify_down(0)
        return item

    def contains(self, map_key):
        return map_key in self._map

    def len(self):
        if len(self._heap) != len(self._map):
            raise ValueError('Invalid state of the priority map')
        return len(self._heap)

    def delete_by_map_key(self, map_key):
        if map_key not in self._map:
            raise ValueError("Map key does not exist")
        heap_index = self._map[map_key]
        item = self._heap[heap_index]
        self._swap(heap_index, len(self._heap) - 1)
        self._heap.pop()
        self._map.pop(map_key)
        if (heap_index < len(self._heap)):
            self._heapify_down(heap_index)
            self._heapify_up(heap_index)
        return item

    def _heapify_up(self, i):
        while i > 0 and self._heap_key(self._heap[i]) > self._heap_key(self._heap[self._parent(i)]):
            self._swap(i, self._parent(i))
            i = self._parent(i)

    def _heapify_down(self, i):
        largest = i
        left = self._left_child(i)
        right = self._right_child(i)

        if left < len(self._heap) and self._heap_key(self._heap[left]) > self._heap_key(self._heap[largest]):
            largest = left

        if right < len(self._heap) and self._heap_key(self._heap[right]) > self._heap_key(self._heap[largest]):
            largest = right

        if largest != i:
            self._swap(i, largest)
            self._heapify_down(largest)

    def _parent(self, i):
        return (i - 1) // 2

    def _left_child(self, i):
        return 2 * i + 1

    def _right_child(self, i):
        return 2 * i + 2

    def _swap(self, i, j):
        self._map[self._map_key(self._heap[i])] = j
        self._map[self._map_key(self._heap[j])] = i
        self._heap[i], self._heap[j] = self._heap[j], self._heap[i]


class StatsEntry:
    def __init__(self, pair, positions):
        self.pair = pair
        self.positions = positions

class TokenizerTrainer:
        def __init__(self, tokens_list, itos, num_merges):
            self.tokens_list = tokens_list
            self.itos = itos
            self._num_merges = num_merges
            self.stoi={}

        def train(self, next_token):
            self._positions=[LinkedArray(tokens) for tokens in self.tokens_list]
            self.get_stats()
            merges_done=0
            print("requested merges",self._num_merges)
            while  merges_done<self._num_merges:
                if self._stats.len() == 0:
                   print("STOPPED: stats became empty")
                   print("Merges completed:", merges_done)
                   break
                merge_stat = self._stats.pop()
                (current_token , next_token) = self.update_itos_and_stoi(next_token, merge_stat)
                #iterate through posistions replace with merge pair
                for position in list(merge_stat.positions):

                    if position not in merge_stat.positions:
                        continue
                    input_index = position[0]
                    token_index = position[1]
                    # print("current_token",current_token)
                    if not self._positions[input_index].exists(token_index):
                        merge_stat.positions.discard(position)
                        continue
                    self.update_right_token(input_index, token_index, merge_stat, current_token)
                    self.update_left_token(input_index, token_index, merge_stat, current_token)
                    self._positions[input_index].replace_pair(token_index, current_token)
                    merge_stat.positions.remove((input_index, token_index))
                merges_done += 1
            return self.itos


        def update_right_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           next_token_idx = position.get_next_index(token_index)
           right_token_idx = position.get_second_next_index(token_index)
           if right_token_idx == None:
               return
           pair = (merge_stat.pair[1], position.get_by_index(right_token_idx))
           self.remove_pair_from_stats(merge_stat, pair, input_index, next_token_idx)
           new_pair = (current_token, pair[1])
           self.add_position_to_pair(new_pair, input_index, token_index)

        def update_left_token(self, input_index, token_index, merge_stat, current_token):
           position = self._positions[input_index]
           left_token_idx = position.get_previous_index(token_index)
           if left_token_idx == None:
               return
           pair = ( position.get_by_index(left_token_idx), merge_stat.pair[0])
           self.remove_pair_from_stats(merge_stat, pair, input_index, left_token_idx)
           new_pair = (pair[0] , current_token)
           self.add_position_to_pair(new_pair, input_index, left_token_idx)

        def add_position_to_pair(self, new_pair, input_index, token_index):
            if self._stats.contains(new_pair):
                stat = self._stats.delete_by_map_key(new_pair)
            else:
                stat = StatsEntry(new_pair, set())

            stat.positions.add((input_index, token_index))
            self._stats.push(stat)


        def remove_pair_from_stats(self, merge_stat, pair, input_index, token_index):
           if pair == merge_stat.pair:
               merge_stat.positions.discard((input_index, token_index))
           else:
                if self._stats.contains(pair):
                    stat = self._stats.delete_by_map_key(pair)
                    stat.positions.discard((input_index, token_index))
                    if len(stat.positions) != 0:
                        self._stats.push(stat)


        def update_itos_and_stoi(self, next_token, merge_stat):
           merge_pair = self.itos[merge_stat.pair[0]] + self.itos[merge_stat.pair[1]]
           if merge_pair in self.stoi:
               current_token = self.stoi[merge_pair]
           else:
               current_token = next_token
               self.itos[current_token] = merge_pair
               self.stoi[merge_pair] = current_token
               next_token = current_token+1
           return  (current_token , next_token)

        def get_stats(self):
             self._stats = MaxPriorityMap(
                                heap_key=lambda item: (len(item.positions),tuple(-x for x in item.pair)),
                                map_key=lambda item: item.pair
                            )
             stats={}
             for  text_id in range(len(self.tokens_list)):
                 tokens = self.tokens_list[text_id]
                 for idx in range(len(tokens)-1):
                     pair = (tokens[idx], tokens[idx+1])
                     if pair not in stats:
                         stats[pair]=StatsEntry(pair, set())
                     stats[pair].positions.add((text_id, idx))
             for key in stats.keys():
                self._stats.push(stats[key])

class TokenNode:
    def __init__(self, byte_value, token, parent, children):
        self.parent = parent
        self.byte_value = byte_value
        self.token = token
        self.children = children

class ToTokensConverter:

    def __init__(self, itos):
        self.itos = itos
        self.unk_token=1
        self._init_token_tree()

    def to_tokens(self, strings):
        return [
            self._to_tokens(string)
            for string in strings
        ]

    def _to_tokens(self, string):

        words = string.split()
        tokens = []
        for word in words:
            data = word.encode("utf-8") + b"</w>"
            word_tokens = self._tokenize_word(data)
            tokens.extend(word_tokens)
        return tokens

    def _tokenize_word(self, data):

        tokens = []

        byte_index = 0

        while byte_index < len(data):

            current_nodes = self._token_roots

            last_node_with_token = None
            last_index_with_token = byte_index

            while byte_index < len(data):

                byte_value = data[byte_index]

                if byte_value in current_nodes:

                    node = current_nodes[byte_value]

                    if node.token is not None:
                        last_node_with_token = node
                        last_index_with_token = byte_index

                    current_nodes = node.children
                    byte_index += 1

                else:
                    break

            if last_node_with_token is None:
                tokens.append(self.unk_token)

                # Move forward by one byte
                byte_index += 1

            else:
                tokens.append(last_node_with_token.token)
                byte_index = last_index_with_token + 1

        return tokens

    def _init_token_tree(self):

        self._token_roots = {}

        for token, token_bytes in self.itos.items():

            if token < 4:
                continue

            current_nodes = self._token_roots
            last_node = None

            for byte_value in token_bytes:

                if byte_value in current_nodes:

                    last_node = current_nodes[byte_value]

                else:

                    last_node = TokenNode(
                        byte_value,
                        None,
                        last_node,
                        {}
                    )

                    current_nodes[byte_value] = last_node

                current_nodes = last_node.children

            if last_node.token is not None:
                raise KeyError("The token is already taken")

            last_node.token = token

class BPETokenizer:
    def __init__(self, num_merges):
        self._num_merges = num_merges
        self.itos = {}
    def get_vocab_size(self):
        return len(self.itos)
    def train(self, texts):
        self.create_base_vocab()

        tokens_list = self.get_ids_from_text(texts)
        # for tokens in tokens_list:
        #     print(tokens)
        trainer = TokenizerTrainer(
                        tokens_list,
                        self.itos,
                        self._num_merges
                    )

        trainer.train(261)

        return self.itos, tokens_list

    def encode(self, text):
        converter = ToTokensConverter(self.itos)
        return converter.to_tokens([text])[0]

    def decode(self, tokens):
        return (
                b"".join(
                    self.itos[token]
                    for token in tokens
                )
                .decode("utf-8", errors="replace")
                .replace("</w>", " ")
                .strip()
            )

    def create_base_vocab(self):
        self.itos = {
            0: b"<pad>",
            1: b"<unk>",
            2: b"<s>",
            3: b"</s>",
            260 : b"</w>"
        }
        for i in range(256):
            self.itos[i + 4] = bytes([i])

    def get_ids_from_text(self, texts):
        tokens_list=[]
        for text in texts:
            words = text.split()
            for word in words:
                word_bytes = [byte + 4 for byte in word.encode("utf-8")]
                word_bytes.append(260)
                tokens_list.append(word_bytes)
        return tokens_list

In [8]:
import numpy as np
import pickle

# Paths
bpe_path = "/kaggle/input/datasets/ksaiveerendraprasad/tokenizer-results/project_bpe_16k.pkl"

# Load BPE tokenizer
with open(bpe_path, "rb") as f:
    tokenizer = pickle.load(f)
print("BPE vocab size:", len(tokenizer.itos))


BPE vocab size: 7999


In [9]:
converter = ToTokensConverter(tokenizer.itos)

In [10]:
import pandas as pd

train_df = pd.DataFrame(train_data)

print("Original training records:", len(train_df))

# Check exact duplicate rows across all columns
duplicate_mask = train_df.duplicated(keep=False)

duplicate_rows = train_df[duplicate_mask].sort_values(
    by=["id"]
)

print("Number of records involved in exact duplicates:", duplicate_mask.sum())
print("Number of extra duplicate rows:", train_df.duplicated().sum())

display(duplicate_rows)

Original training records: 7030
Number of records involved in exact duplicates: 0
Number of extra duplicate rows: 0


,id,document,summary


In [11]:
train_df = train_df.drop_duplicates(
    subset=["id", "document", "summary"],
    keep="first"
).reset_index(drop=True)

print("Training records after removing duplicates:", len(train_df))

Training records after removing duplicates: 7030


In [12]:
train_df.head()

,id,document,summary
0,6712,IVIL Appeal No.768 (NT) of 1977 etc. From the ...,"The appellant, a manufacturer of cement, enter..."
1,2858,Appeal No. 36 of 1967. Appeal by special leave...,The appellant executed a usufructuary mortgage...
2,6466,ivil Appeal No. 2072 of 1990. From the Judgmen...,The appellant had filed a suit in the High Cou...
3,6818,ivil Appeal No. 3504 of 1991. the Judgment and...,The appellant was injured in a road accident o...
4,5824,Civil Appeal No. 3189/1989. From the Judgment ...,Some persons were plying their business by squ...


In [13]:
from tqdm.auto import tqdm

def get_token_length(text):
    return len(
        converter.to_tokens([text])[0]
    )

tqdm.pandas()

train_df["document_tokens"] = train_df["document"].progress_apply(
    get_token_length
)

train_df["summary_tokens"] = train_df["summary"].progress_apply(
    get_token_length
)

  0%|          | 0/7030 [00:00<?, ?it/s]

  0%|          | 0/7030 [00:00<?, ?it/s]

In [14]:
print("DOCUMENT TOKEN STATISTICS")
display(train_df["document_tokens"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

print("SUMMARY TOKEN STATISTICS")
display(train_df["summary_tokens"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

DOCUMENT TOKEN STATISTICS


,document_tokens
count,7030.000000
mean,5467.621764
std,6561.059632
min,185.000000
25%,2485.500000
50%,3887.500000
75%,6200.000000
90%,9884.100000
95%,14173.550000
99%,30976.570000


SUMMARY TOKEN STATISTICS


,summary_tokens
count,7030.000000
mean,1071.220199
std,1182.218137
min,0.000000
25%,526.000000
50%,799.500000
75%,1223.750000
90%,1927.100000
95%,2604.000000
99%,5580.470000


In [15]:
print("Total training records:", len(train_df))

print("Maximum document tokens:", train_df["document_tokens"].max())
print("Maximum summary tokens:", train_df["summary_tokens"].max())

print("Documents exceeding 1024 tokens:",
      (train_df["document_tokens"] > 1024).sum())

print("Documents exceeding 2048 tokens:",
      (train_df["document_tokens"] > 2048).sum())

print("Documents exceeding 4096 tokens:",
      (train_df["document_tokens"] > 4096).sum())

Total training records: 7030
Maximum document tokens: 177759
Maximum summary tokens: 36327
Documents exceeding 1024 tokens: 6831
Documents exceeding 2048 tokens: 5855
Documents exceeding 4096 tokens: 3284


In [16]:
# Top 10 longest documents
display(
    train_df.nlargest(10, "document_tokens")[
        ["id", "document_tokens", "summary_tokens"]
    ]
)

# Top 10 longest summaries
display(
    train_df.nlargest(10, "summary_tokens")[
        ["id", "document_tokens", "summary_tokens"]
    ]
)

,id,document_tokens,summary_tokens
5656,3498,177759,36327
37,34,113610,2881
2850,4656,112218,11766
756,67,106873,3250
1830,2189,103701,11700
6183,6581,87701,18947
5532,568,76395,2269
6579,5712,71665,16900
252,1861,68419,7849
6683,6310,67984,13181


,id,document_tokens,summary_tokens
5656,3498,177759,36327
542,4267,66312,23071
6183,6581,87701,18947
3721,4740,48389,18707
2687,5031,64586,17514
6579,5712,71665,16900
6683,6310,67984,13181
3961,4861,32096,12978
1054,6932,48526,12126
190,5217,39038,12065


In [17]:
display(
    train_df.nsmallest(10, "document_tokens")[
        ["id", "document_tokens", "summary_tokens"]
    ]
)

display(
    train_df.nsmallest(10, "summary_tokens")[
        ["id", "document_tokens", "summary_tokens"]
    ]
)

,id,document_tokens,summary_tokens
1345,5493,185,1158
2692,4822,257,139
570,6175,263,98
1482,5137,268,194
3890,5156,290,178
4689,4579,304,119
1625,4057,306,142
3586,4824,310,138
1248,5247,319,207
6911,4799,325,0


,id,document_tokens,summary_tokens
2950,299,2739,0
6911,4799,325,0
5505,265,1913,42
1304,262,835,45
608,4545,538,55
4808,87,3752,56
1911,184,2794,60
2633,4713,545,67
5134,158,6542,67
333,317,4070,73


In [18]:
train_df["summary_document_ratio"] = (
    train_df["summary_tokens"] /
    train_df["document_tokens"].replace(0, float("nan"))
)

display(
    train_df.nlargest(20, "summary_document_ratio")[
        ["id", "document_tokens", "summary_tokens",
         "summary_document_ratio"]
    ]
)

,id,document_tokens,summary_tokens,summary_document_ratio
1345,5493,185,1158,6.259459
2554,5676,550,2141,3.892727
4112,6856,537,422,0.785847
6041,4216,1329,989,0.744169
6411,4793,2154,1593,0.739554
1652,6060,922,674,0.731020
1482,5137,268,194,0.723881
5949,4210,2056,1484,0.721790
5123,6956,1516,1075,0.709103
1980,4645,777,546,0.702703


In [19]:

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ============================================================
# 1. CREATE DATAFRAME
# ============================================================

train_df = pd.DataFrame(train_data).copy()

print("Original records:", len(train_df))


# ============================================================
# 2. REMOVE EXACT DUPLICATE ROWS
# ============================================================

before = len(train_df)

train_df = train_df.drop_duplicates(
    subset=["id", "document", "summary"],
    keep="first"
).reset_index(drop=True)

print("Exact duplicate rows removed:", before - len(train_df))


# ============================================================
# 3. REMOVE EMPTY DOCUMENTS AND SUMMARIES
# ============================================================

empty_document = (
    train_df["document"].isna() |
    train_df["document"].fillna("").str.strip().eq("")
)

empty_summary = (
    train_df["summary"].isna() |
    train_df["summary"].fillna("").str.strip().eq("")
)

print("Empty documents:", empty_document.sum())
print("Empty summaries:", empty_summary.sum())

train_df = train_df.loc[
    ~(empty_document | empty_summary)
].copy().reset_index(drop=True)


# ============================================================
# 4. CALCULATE TOKEN LENGTHS
# ============================================================

def get_token_length(text):
    return len(
        converter.to_tokens([text])[0]
    )

train_df["document_tokens"] = train_df["document"].progress_apply(
    get_token_length
)

train_df["summary_tokens"] = train_df["summary"].progress_apply(
    get_token_length
)


# ============================================================
# 5. FILTER SHORT DOCUMENTS
# ============================================================

MIN_DOCUMENT_TOKENS = 128

short_document = (
    train_df["document_tokens"] < MIN_DOCUMENT_TOKENS
)

print("Short documents removed:", short_document.sum())


# ============================================================
# 6. FILTER LONG SUMMARIES
# ============================================================

MAX_SUMMARY_TOKENS = 6000

long_summary = (
    train_df["summary_tokens"] > MAX_SUMMARY_TOKENS
)

print("Long summaries removed:", long_summary.sum())


# ============================================================
# 7. FILTER SUMMARY/DOCUMENT RATIO
# ============================================================

MAX_SUMMARY_DOCUMENT_RATIO = 0.25

train_df["summary_document_ratio"] = (
    train_df["summary_tokens"] /
    train_df["document_tokens"]
)

long_relative_summary = (
    train_df["summary_document_ratio"] >
    MAX_SUMMARY_DOCUMENT_RATIO
)

print("Summary/document ratio > 0.25:",
      long_relative_summary.sum())


# ============================================================
# 8. APPLY FILTERING
# ============================================================

remove_mask = (
    short_document |
    long_summary |
    long_relative_summary
)

train_clean = (
    train_df.loc[~remove_mask]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

print("\n========== FINAL DATASET ==========")

print("Records after duplicate removal and empty filtering:",
      len(train_df))

print("Records removed by length filters:",
      remove_mask.sum())

print("Final training records:", len(train_clean))

print("\nDocument token statistics:")
display(train_clean["document_tokens"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

print("\nSummary token statistics:")
display(train_clean["summary_tokens"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

print("\nSummary/document ratio statistics:")
display(train_clean["summary_document_ratio"].describe())


Original records: 7030
Exact duplicate rows removed: 0
Empty documents: 0
Empty summaries: 2


  0%|          | 0/7028 [00:00<?, ?it/s]

  0%|          | 0/7028 [00:00<?, ?it/s]

Short documents removed: 0
Long summaries removed: 58
Summary/document ratio > 0.25: 2910

========== FINAL DATASET ==========
Records after duplicate removal and empty filtering: 7028
Records removed by length filters: 2942
Final training records: 4086

Document token statistics:


,document_tokens
count,4086.000000
mean,6343.744983
std,5940.279205
min,377.000000
25%,3211.250000
50%,4751.500000
75%,7358.250000
90%,11571.000000
95%,16320.750000
99%,30331.900000



Summary token statistics:


,summary_tokens
count,4086.000000
mean,882.780715
std,696.210581
min,42.000000
25%,462.000000
50%,686.500000
75%,1051.750000
90%,1648.000000
95%,2144.250000
99%,3803.100000



Summary/document ratio statistics:


count    4086.000000
mean        0.154299
std         0.056257
min         0.010242
25%         0.109593
50%         0.156637
75%         0.201551
max         0.250000
Name: summary_document_ratio, dtype: float64

In [20]:
import pandas as pd
from tqdm.auto import tqdm

# Load test data
test_df = pd.DataFrame(test_data).copy()

print("Total test records:", len(test_df))

# Calculate document token lengths
tqdm.pandas()

def get_token_length(text):
    if pd.isna(text):
        return 0

    return len(
        converter.to_tokens([text])[0]
    )

test_df["document_tokens"] = test_df["document"].progress_apply(
    get_token_length
)

# Token length statistics
print("\nTEST DOCUMENT TOKEN STATISTICS")

display(
    test_df["document_tokens"].describe(
        percentiles=[
            0.25, 0.50, 0.75,
            0.90, 0.95, 0.99
        ]
    ).to_frame()
)

Total test records: 100


  0%|          | 0/100 [00:00<?, ?it/s]


TEST DOCUMENT TOKEN STATISTICS


,document_tokens
count,100.000000
mean,5991.870000
std,6852.554134
min,662.000000
25%,2635.750000
50%,3883.500000
75%,7015.000000
90%,11359.400000
95%,17040.850000
99%,36892.180000


In [21]:
import json

train_clean.to_json(
    "train_cleaned.json",
    orient="records",
    indent=2,
    force_ascii=False
)

In [22]:
import pandas as pd
import numpy as np

# Token length comparison
comparison = pd.DataFrame({
    "Train": train_clean["document_tokens"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ),
    "Test": test_df["document_tokens"].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
})

display(comparison)

,Train,Test
count,4086.000000,100.000000
mean,6343.744983,5991.870000
std,5940.279205,6852.554134
min,377.000000,662.000000
25%,3211.250000,2635.750000
50%,4751.500000,3883.500000
75%,7358.250000,7015.000000
90%,11571.000000,11359.400000
95%,16320.750000,17040.850000
99%,30331.900000,36892.180000


In [23]:
thresholds = [1024, 2048, 4096, 8192, 16384, 32768]

results = []

for threshold in thresholds:

    train_count = (
        train_clean["document_tokens"] > threshold
    ).sum()

    test_count = (
        test_df["document_tokens"] > threshold
    ).sum()

    results.append({
        "Context Length": threshold,
        "Train Exceeding": train_count,
        "Train %": round(train_count / len(train_clean) * 100, 2),
        "Test Exceeding": test_count,
        "Test %": round(test_count / len(test_df) * 100, 2)
    })

threshold_comparison = pd.DataFrame(results)

display(threshold_comparison)

,Context Length,Train Exceeding,Train %,Test Exceeding,Test %
0,1024,4066,99.51,96,96.0
1,2048,3803,93.07,87,87.0
2,4096,2446,59.86,47,47.0
3,8192,830,20.31,15,15.0
4,16384,202,4.94,7,7.0
5,32768,29,0.71,2,2.0


In [24]:
CHUNK_SIZE = 1024
CHUNK_OVERLAP = 128
STRIDE = CHUNK_SIZE - CHUNK_OVERLAP

def get_num_chunks(length):
    if length <= CHUNK_SIZE:
        return 1

    return 1 + (
        (length - CHUNK_SIZE + STRIDE - 1) // STRIDE
    )

train_clean["num_chunks"] = (
    train_clean["document_tokens"].apply(get_num_chunks)
)

test_df["num_chunks"] = (
    test_df["document_tokens"].apply(get_num_chunks)
)

print("TRAIN CHUNK STATISTICS")
display(train_clean["num_chunks"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

print("TEST CHUNK STATISTICS")
display(test_df["num_chunks"].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).to_frame())

TRAIN CHUNK STATISTICS


,num_chunks
count,4086.000000
mean,7.440529
std,6.639540
min,1.000000
25%,4.000000
50%,6.000000
75%,9.000000
90%,13.000000
95%,19.000000
99%,34.000000


TEST CHUNK STATISTICS


,num_chunks
count,100.000000
mean,7.020000
std,7.646191
min,1.000000
25%,3.000000
50%,5.000000
75%,8.000000
90%,13.100000
95%,19.100000
99%,41.140000


In [25]:
import json
import torch
from torch.utils.data import Dataset, DataLoader

MAX_CHUNK_LEN = 1024
CHUNK_OVERLAP = 64
MAX_CHUNKS = 4


class CJPDataset(Dataset):
    def __init__(
        self,
        json_path,
        converter = converter,
        max_chunk_len=MAX_CHUNK_LEN,
        overlap=CHUNK_OVERLAP,
        max_chunks=MAX_CHUNKS,
        has_labels=True,
    ):
        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.converter = converter
        self.max_chunk_len = max_chunk_len
        self.overlap = overlap
        self.max_chunks = max_chunks
        self.has_labels = has_labels
        
        assert 0 <= overlap < max_chunk_len
        assert max_chunk_len >= 3, "Need room for <s>, content, and </s>"

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Your custom tokenizer should return a list of token IDs.
        token_ids = self.converter.to_tokens([item["text"]])[0]

        # IDs from your itos vocabulary.
        pad_id = 0
        unk_id = 1
        bos_id = 2  # <s>
        eos_id = 3  # </s>

        # Reserve 2 positions for <s> and </s>.
        payload_len = self.max_chunk_len - 2
        step = payload_len - self.overlap

        chunks = []

        for start in range(0, len(token_ids), step):
            content = token_ids[start:start + payload_len]

            if not content:
                break

            chunk_ids = [bos_id] + content + [eos_id]
            chunks.append(chunk_ids)

            if start + payload_len >= len(token_ids):
                break

            if len(chunks) >= self.max_chunks:
                break

        # Handle an empty document.
        if not chunks:
            chunks = [[bos_id, eos_id]]

        return {
            "id": item["id"],
            "chunks": chunks,
            "label": int(item["label"]) if self.has_labels else -1,
        }

In [26]:

import json
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

MAX_CHUNK_LEN = 1024
CHUNK_OVERLAP = 128
MAX_CHUNKS = 20

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3


class LegalSummarizationDataset(Dataset):

    def __init__(
        self,
        json_path,
        converter,
        max_chunk_len=MAX_CHUNK_LEN,
        overlap=CHUNK_OVERLAP,
        max_chunks=MAX_CHUNKS,
        has_labels=True,
    ):

        with open(json_path, "r", encoding="utf-8") as f:
            self.data = json.load(f)

        self.converter = converter
        self.max_chunk_len = max_chunk_len
        self.overlap = overlap
        self.max_chunks = max_chunks
        self.has_labels = has_labels

        assert 0 <= overlap < max_chunk_len - 2
        assert max_chunk_len >= 3

    def __len__(self):
        return len(self.data)

    def tokenize(self, text):
        return self.converter.to_tokens([text])[0]

    def create_chunks(self, token_ids):

        payload_len = self.max_chunk_len - 2
        step = payload_len - self.overlap

        chunks = []

        for start in range(0, len(token_ids), step):

            content = token_ids[start:start + payload_len]

            if not content:
                break

            chunk = [BOS_ID] + content + [EOS_ID]
            chunks.append(chunk)

            if start + payload_len >= len(token_ids):
                break

        if not chunks:
            chunks = [[BOS_ID, EOS_ID]]

        # Select chunks distributed across the document
        if len(chunks) > self.max_chunks:

            indices = np.linspace(
                0,
                len(chunks) - 1,
                self.max_chunks,
                dtype=int
            )

            chunks = [chunks[i] for i in indices]

        return chunks

    def __getitem__(self, idx):

        item = self.data[idx]

        document_ids = self.tokenize(item["document"])

        chunks = self.create_chunks(document_ids)

        summary_ids = None

        if self.has_labels:

            summary_tokens = self.tokenize(item["summary"])

            summary_ids = (
                [BOS_ID] + summary_tokens + [EOS_ID]
            )

        return {
            "id": item["id"],
            "chunks": chunks,
            "num_chunks": len(chunks),
            "document_tokens": len(document_ids),
            "summary_ids": summary_ids,
        }


In [27]:

def summarization_collate_fn(batch):

    batch_size = len(batch)

    pad_id = PAD_ID

    # ========================================================
    # 1. PAD DOCUMENT CHUNKS
    # ========================================================

    input_ids = torch.full(
        (batch_size, MAX_CHUNKS, MAX_CHUNK_LEN),
        pad_id,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, MAX_CHUNKS, MAX_CHUNK_LEN),
        dtype=torch.long
    )

    chunk_mask = torch.zeros(
        (batch_size, MAX_CHUNKS),
        dtype=torch.bool
    )

    for i, item in enumerate(batch):

        for j, chunk in enumerate(item["chunks"]):

            chunk_len = len(chunk)

            input_ids[i, j, :chunk_len] = torch.tensor(
                chunk,
                dtype=torch.long
            )

            attention_mask[i, j, :chunk_len] = 1

            chunk_mask[i, j] = True

    # ========================================================
    # 2. PAD REFERENCE SUMMARIES
    # ========================================================

    summary_ids = None
    summary_mask = None

    if batch[0]["summary_ids"] is not None:

        max_summary_len = max(
            len(item["summary_ids"])
            for item in batch
        )

        summary_ids = torch.full(
            (batch_size, max_summary_len),
            pad_id,
            dtype=torch.long
        )

        summary_mask = torch.zeros(
            (batch_size, max_summary_len),
            dtype=torch.long
        )

        for i, item in enumerate(batch):

            ids = item["summary_ids"]
            summary_len = len(ids)

            summary_ids[i, :summary_len] = torch.tensor(
                ids,
                dtype=torch.long
            )

            summary_mask[i, :summary_len] = 1

    # ========================================================
    # 3. RETURN BATCH
    # ========================================================

    return {
        "ids": [item["id"] for item in batch],

        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "chunk_mask": chunk_mask,

        "summary_ids": summary_ids,
        "summary_mask": summary_mask,

        "num_chunks": torch.tensor(
            [item["num_chunks"] for item in batch],
            dtype=torch.long
        ),

        "document_tokens": torch.tensor(
            [item["document_tokens"] for item in batch],
            dtype=torch.long
        )
    }


In [28]:
# train_dataset = LegalSummarizationDataset(
#     json_path="train_cleaned.json",
#     converter=converter,
#     max_chunk_len=1024,
#     overlap=128,
#     max_chunks=20,
#     has_labels=True
# )

# train_loader = DataLoader(
#     train_dataset,
#     batch_size=2,
#     shuffle=True,
#     collate_fn=summarization_collate_fn,
#     num_workers=0,
#     pin_memory=True
# )

In [29]:
# batch = next(iter(train_loader))

# for key, value in batch.items():
#     if isinstance(value, torch.Tensor):
#         print(f"{key}: {value.shape}, {value.dtype}")
#     else:
#         print(f"{key}: {value}")

In [30]:
# # Number of valid chunks per document
# print("Number of chunks:", batch["num_chunks"])

# # Valid tokens in each chunk
# print("Chunk token counts:")
# print(batch["attention_mask"].sum(dim=-1))

# # Valid summary tokens
# print("Summary token counts:")
# print(batch["summary_mask"].sum(dim=-1))

# # Check padding
# print(
#     "Padding positions correctly masked:",
#     (batch["attention_mask"] == 0).sum().item()
# )

# # Verify chunk limits
# assert batch["input_ids"].shape[-1] == 1024
# assert batch["input_ids"].shape[1] == 20
# assert batch["num_chunks"].max() <= 20

# # Verify summary boundaries
# for i in range(len(batch["ids"])):
#     length = batch["summary_mask"][i].sum().item()

#     assert batch["summary_ids"][i, 0].item() == BOS_ID
#     assert batch["summary_ids"][i, length - 1].item() == EOS_ID

# print("All sanity checks passed!")

In [31]:

"""Rotary Positional Embeddings, implemented from scratch."""
import torch


def build_rope_cache(max_seq_len: int, d_head: int, base: float = 10000.0):
    """
    Returns (cos, sin) tensors of shape (max_seq_len, d_head) ready to be applied to
    query/key tensors of head dimension d_head. d_head must be even. This scaffold uses
    the split-half rotate_half convention: channel i is paired with channel i+d_head/2.
    """
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inverse_frequencies =torch.tensor([1/base ** (2*i/d_head) for i in range(d_head//2)],dtype = torch.float32)
    positions = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(positions, inverse_frequencies)
    angles = torch.cat([freqs, freqs], dim=-1)
    return angles.cos(), angles.sin()



def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split the last dim in half and return (-x2, x1) concatenated -- the 90-degree
    rotation step used by the "rotate_half" formulation of RoPE."""
    # TODO
    d_head = x.shape[-1]
    x1, x2 = x[...,:d_head//2], x[...,d_head//2:]
    return torch.cat([-x2, x1], dim=-1)



def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x: (B, n_heads, T, d_head) -- a query or key tensor.
    cos, sin: (T, d_head), sliced from build_rope_cache's output for the current T.
    Returns the rotated tensor, same shape as x.
    Formula: x_rotated = x * cos + rotate_half(x) * sin
    """
    # TODO: broadcast cos/sin to (1, 1, T, d_head) and apply the formula above
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    x_rotated = x*cos + rotate_half(x)*sin
    return x_rotated



In [32]:

# TODO 3.2: integrate RoPE through attention and the decoder architecture in this section.
"""Attention supporting MHA/MQA/GQA (Q2) with a choice of positional scheme: RoPE applied
to q/k inside attention, or none (if positions are handled by sinusoidal PE at the input,
as in Q1)."""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, use_rope=False, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.use_rope = use_rope
        # Preserve the Q1.4 hook. None -> standard 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)


        if use_rope:
            cos, sin = build_rope_cache(max_seq_len, self.d_head)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x: torch.Tensor,attn_mask=None) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).reshape(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).reshape(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.use_rope:
            # TODO: apply_rope to q and k (NOT v) using self.rope_cos[:T], self.rope_sin[:T]
            q = apply_rope(q, self.rope_cos[:T], self.rope_sin[:T])
            k = apply_rope(k, self.rope_cos[:T], self.rope_sin[:T])
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim = 1)
            v = v.repeat_interleave(self.n_rep, dim = 1)
       
        out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.attn_dropout.p if self.training else 0.0
        )
        out = out.transpose(1, 2).reshape(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out



In [33]:

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F



@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None      # preserve Q1.4 hook



def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    PE = torch.zeros(max_len, d_model)
    for pos in range(max_len):
      for i in range(d_model//2):
        PE[pos, 2*i] = math.sin(pos / 10000**(2*i/d_model))
        PE[pos, 2*i+1] = math.cos(pos / 10000**(2*i/d_model))
    return PE



def build_attn_mask(is_causal, padding_mask, T):

    if padding_mask is not None:
        B = padding_mask.size(0)
        device = padding_mask.device
    elif torch.is_tensor(is_causal):
        B = is_causal.numel()
        device = is_causal.device
    else:
        B = 1
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Convert is_causal into a Boolean tensor of shape (B,)
    if is_causal is None:
        is_causal = False

    if isinstance(is_causal, bool):
        is_causal = torch.full(
            (B,),
            is_causal,
            dtype=torch.bool,
            device=device
        )
    else:
        is_causal = is_causal.to(
            device=device,
            dtype=torch.bool
        ).reshape(-1)

    if is_causal.numel() != B:
        raise ValueError("is_causal must contain one value per batch item")

    full = torch.ones(
        T, T,
        dtype=torch.bool,
        device=device
    )

    causal = torch.tril(full)

    # True = allowed attention
    allow = torch.where(
        is_causal[:, None, None],
        causal.unsqueeze(0),
        full.unsqueeze(0)
    )

    if padding_mask is not None:

        # padding_mask: (B,T), True = valid token
        allow = allow & padding_mask[:, None, :]

    # Output: (B,1,T,T)
    return allow.unsqueeze(1)

    
class EncoderFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout)
        )   # unchanged from Question 1

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        # TODO: pass use_rope=(cfg.pos_emb_type == "rope") to CausalSelfAttention
        self.attn = MultiHeadAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                        cfg.dropout, use_rope=(cfg.pos_emb_type == "rope"),
                                        attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = EncoderFeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, attn_mask=None):
        x = x + self.attn(self.ln1(x), attn_mask)
        x = x + self.ffn(self.ln2(x))
        return x   # unchanged from Question 1


class EncoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)
        # NOTE: when cfg.pos_emb_type == "rope", we deliberately do NOT build a
        # sinusoidal table at all -- position information is injected inside every
        # attention layer instead of once at the input.
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(
            cfg.d_model,
            cfg.vocab_size,
            bias=False
        )
        
        self.lm_head.weight = self.tok_emb.weight
                

    def forward(self, idx, padding_mask=None, is_causal=None):                 # CHANGED: is_causal
            B, T = idx.shape
            if padding_mask is not None:
                if padding_mask.shape != (B, T):
                    raise ValueError("padding_mask must have shape (B, T)")
                padding_mask = padding_mask.to(device=idx.device, dtype=torch.bool)
            if is_causal is None:                                                  # old behaviour: bidirectional
                is_causal = torch.zeros(B, dtype=torch.bool, device=idx.device)
            attn_mask = build_attn_mask(is_causal, padding_mask, T)
            x = self.tok_emb(idx)
            if self.cfg.pos_emb_type == "sinusoidal":
                x = x + self.pos_emb[:T].unsqueeze(0)
            for block in self.blocks:
                x = block(x, attn_mask=attn_mask)
            x = self.ln_f(x)
            return self.lm_head(x) 


In [34]:

class LoRMALinear(nn.Module):

    def __init__(
        self,
        layer,
        rank=16,
        alpha=32,
        dropout=0.05,
        variant="plus"
    ):
        super().__init__()

        if variant not in {"plus", "pi"}:
            raise ValueError("variant must be 'plus' or 'pi'")

        self.layer = layer
        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank
        self.variant = variant
        self.dropout = nn.Dropout(dropout)

        for p in self.layer.parameters():
            p.requires_grad = False

        d = layer.out_features

        self.A = nn.Parameter(torch.empty(rank, d))
        self.B = nn.Parameter(torch.empty(d, rank))

        if variant == "plus":

            nn.init.kaiming_uniform_(
                self.A,
                a=math.sqrt(5)
            )

            nn.init.zeros_(self.B)

        else:

            nn.init.normal_(self.B)

            with torch.no_grad():
                self.B[:, 0] = 1.0
                self.A.zero_()
                self.A[0, 0] = 1.0

    @staticmethod
    def inflate_pi(BA):

        d = BA.shape[0]

        return torch.stack(
            [
                torch.roll(BA[i], shifts=i, dims=0)
                for i in range(d)
            ],
            dim=0
        )

    def forward(self, x):

        base = self.layer(x)
        z = self.dropout(base)

        BA = self.B @ self.A

        if self.variant == "plus":

            adapted = F.linear(z, BA)

        else:

            BA_pi = self.inflate_pi(BA)
            adapted = F.linear(z, BA_pi)

        return base + self.scale * adapted


In [35]:

class LoRMAEncoder(nn.Module):

    def __init__(
        self,
        base_model,
        rank=16,
        alpha=32,
        lorma_dropout=0.05,
        variant="plus",
        projections=(
            "q_proj",
            "k_proj",
            "v_proj",
            "out_proj"
        )
    ):
        super().__init__()

        self.model = base_model
        self.rank = rank
        self.alpha = alpha
        self.lorma_dropout = lorma_dropout
        self.variant = variant
        self.projections = projections

        # Freeze pretrained model
        for param in self.model.parameters():
            param.requires_grad = False

        self._add_lorma()

    def _wrap_attention(self, attention):

        for proj_name in self.projections:

            layer = getattr(attention, proj_name)

            if not isinstance(layer, nn.Linear):
                raise TypeError(
                    f"{proj_name} is not nn.Linear: {type(layer)}"
                )

            setattr(
                attention,
                proj_name,
                LoRMALinear(
                    layer=layer,
                    rank=self.rank,
                    alpha=self.alpha,
                    dropout=self.lorma_dropout,
                    variant=self.variant
                )
            )

    def _add_lorma(self):

        # Support EncoderLM and EncoderDecoderTransformer
        if hasattr(self.model, "encoder"):
            encoder = self.model.encoder
        else:
            encoder = self.model

        for block in encoder.blocks:
            self._wrap_attention(block.attn)

    def forward(self, *args, **kwargs):
        return self.model(*args, **kwargs)

    def get_trainable_parameters(self):

        return [
            p for p in self.parameters()
            if p.requires_grad
        ]

    def print_trainable_parameters(self):

        trainable = sum(
            p.numel()
            for p in self.parameters()
            if p.requires_grad
        )

        total = sum(
            p.numel()
            for p in self.parameters()
        )

        print(f"Trainable parameters: {trainable:,}")
        print(f"Total parameters:     {total:,}")

        print(
            f"Trainable percentage: "
            f"{100 * trainable / total:.4f}%"
        )


In [36]:
cfg = LMConfig(
    vocab_size=VOCAB_SIZE,
    d_model=MODEL_DIM,
    n_layers=10,
    n_heads=N_HEADS,
    n_kv_heads=N_KV_HEADS,
    d_ff=MODEL_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=0.1,
    pos_emb_type="rope",
)

model_phase1 = EncoderLM(cfg)

In [37]:
checkpoint = torch.load(
    "/kaggle/input/datasets/ksaiveerendraprasad/hybridencoder-model/hybrid_encoder.pt",
    map_location=device,
    weights_only=False
)
state_dict = checkpoint["model_state_dict"]

new_state_dict = {}

for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    new_state_dict[key] = value

model_phase1.load_state_dict(new_state_dict)

<All keys matched successfully>

In [38]:
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Special token IDs from your custom tokenizer
PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

# Chunk configuration (same as your existing dataset)
MAX_CHUNK_LEN = 1024
CHUNK_OVERLAP = 128
MAX_CHUNKS = 20

PAYLOAD_LEN = MAX_CHUNK_LEN - 2
STRIDE = PAYLOAD_LEN - CHUNK_OVERLAP

print("Device:", DEVICE)
print("Payload length:", PAYLOAD_LEN)
print("Stride:", STRIDE)

Device: cuda
Payload length: 1022
Stride: 894


In [39]:
def split_summary_into_sentences(text):
    text = str(text).strip()

    if not text:
        return []

    sentences = re.split(r'(?<=[.!?])\s+', text)

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]

In [40]:
model_phase1 = model_phase1.to(DEVICE)
model_phase1.eval()

@torch.no_grad()
def get_text_embedding(token_ids):
    """
    Generate one embedding for a text sequence using
    the Phase 1 Decoder's final hidden representations.
    """

    token_ids = token_ids[:PAYLOAD_LEN]

    if len(token_ids) == 0:
        return None

    input_ids = torch.tensor(
        [[BOS_ID] + token_ids + [EOS_ID]],
        dtype=torch.long,
        device=DEVICE
    )

    # Token embeddings
    x = model_phase1.tok_emb(input_ids)

    # Decoder blocks
    for block in model_phase1.blocks:
        x, _ = block(x)

    # Final layer normalization
    hidden = model_phase1.ln_f(x)

    # Exclude BOS and EOS
    token_hidden = hidden[:, 1:-1, :]

    # Mean pooling
    embedding = token_hidden.mean(dim=1)

    # Normalize for cosine similarity
    embedding = F.normalize(embedding, p=2, dim=-1)

    return embedding.squeeze(0)

In [41]:
@torch.no_grad()
def align_summary_to_chunks(document, summary, dataset_item):
    """
    Assign each summary sentence to its most similar document chunk.

    dataset_item is obtained from your existing train_dataset,
    ensuring we reuse exactly the same chunking and chunk selection.
    """

    chunks = dataset_item["chunks"]

    sentences = split_summary_into_sentences(summary)

    if not sentences:
        return [""] * len(chunks), []

    # --------------------------------------------------
    # 1. Get document chunk embeddings
    # --------------------------------------------------

    chunk_embeddings = []

    for chunk in chunks:
        # Remove BOS and EOS before embedding
        content_ids = chunk[1:-1]

        embedding = get_text_embedding(content_ids)

        if embedding is not None:
            chunk_embeddings.append(embedding)

    if not chunk_embeddings:
        return [""] * len(chunks), []

    chunk_embeddings = torch.stack(chunk_embeddings)

    # --------------------------------------------------
    # 2. Get summary sentence embeddings
    # --------------------------------------------------

    sentence_embeddings = []
    valid_sentences = []

    for sentence in sentences:

        sentence_ids = converter.to_tokens([sentence])[0]

        # Very long sentences are truncated to the model's
        # available payload length.
        embedding = get_text_embedding(sentence_ids)

        if embedding is not None:
            sentence_embeddings.append(embedding)
            valid_sentences.append(sentence)

    if not sentence_embeddings:
        return [""] * len(chunks), []

    sentence_embeddings = torch.stack(sentence_embeddings)

    # --------------------------------------------------
    # 3. Cosine similarity
    # --------------------------------------------------

    similarity = sentence_embeddings @ chunk_embeddings.T

    # --------------------------------------------------
    # 4. Assign each sentence to its best chunk
    # --------------------------------------------------

    best_chunks = similarity.argmax(dim=1).cpu().tolist()
    best_scores = similarity.max(dim=1).values.cpu().tolist()

    aligned_targets = [""] * len(chunks)

    for sentence, chunk_idx in zip(valid_sentences, best_chunks):

        if aligned_targets[chunk_idx]:
            aligned_targets[chunk_idx] += " "

        aligned_targets[chunk_idx] += sentence

    alignment_info = [
        {
            "sentence": sentence,
            "chunk_index": int(chunk_idx),
            "similarity": float(score)
        }
        for sentence, chunk_idx, score
        in zip(valid_sentences, best_chunks, best_scores)
    ]

    return aligned_targets, alignment_info

In [42]:
# aligned_train_data = []

# for idx in tqdm(range(len(train_dataset))):

#     item = train_dataset[idx]

#     document = train_clean.iloc[idx]["document"]
#     summary = train_clean.iloc[idx]["summary"]

#     aligned_targets, alignment_info = align_summary_to_chunks(
#         document=document,
#         summary=summary,
#         dataset_item=item
#     )

#     aligned_train_data.append({
#         "id": str(item["id"]),
#         "document": document,
#         "summary": summary,
#         "chunks": item["chunks"],
#         "chunk_targets": aligned_targets,
#         "alignment_info": alignment_info
#     })

# print("Processed documents:", len(aligned_train_data))

In [43]:
with open("/kaggle/input/datasets/ksaiveerendraprasad/train-chunk-aligned/train_chunk_aligned.json", "r", encoding="utf-8") as f:
    aligned_train_data = json.load(f)

print("Documents loaded:", len(aligned_train_data))
print("Sample keys:", aligned_train_data[0].keys())

Documents loaded: 4086
Sample keys: dict_keys(['id', 'document', 'summary', 'chunks', 'chunk_targets', 'alignment_info'])


In [44]:
# num_chunks_per_doc = []
# num_supervised_chunks = []
# num_aligned_sentences = []
# all_similarity_scores = []

# for item in aligned_train_data:

#     targets = item["chunk_targets"]

#     num_chunks_per_doc.append(len(targets))

#     supervised = sum(bool(x.strip()) for x in targets)
#     num_supervised_chunks.append(supervised)

#     num_aligned_sentences.append(
#         len(item["alignment_info"])
#     )

#     all_similarity_scores.extend(
#         x["similarity"] for x in item["alignment_info"]
#     )

# print("Documents:", len(aligned_train_data))

# print(
#     "Average chunks per document:",
#     np.mean(num_chunks_per_doc)
# )

# print(
#     "Average supervised chunks per document:",
#     np.mean(num_supervised_chunks)
# )

# print(
#     "Average aligned sentences per document:",
#     np.mean(num_aligned_sentences)
# )

# print(
#     "Average cosine similarity:",
#     np.mean(all_similarity_scores)
# )

# print(
#     "Median cosine similarity:",
#     np.median(all_similarity_scores)
# )

In [45]:
alignment_df = pd.DataFrame([
    {
        "similarity": info["similarity"],
        "chunk_index": info["chunk_index"],
        "sentence": info["sentence"],
        "document_id": doc["id"]
    }
    for doc in aligned_train_data
    for info in doc["alignment_info"]
])

print(alignment_df["similarity"].describe())

print("\nSimilarity distribution:")
print(
    pd.cut(
        alignment_df["similarity"],
        bins=[0, 0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
    ).value_counts().sort_index()
)

count    117989.000000
mean          0.734291
std           0.191158
min           0.078253
25%           0.612675
50%           0.802864
75%           0.886346
max           0.987138
Name: similarity, dtype: float64

Similarity distribution:
similarity
(0.0, 0.3]     3711
(0.3, 0.5]    14000
(0.5, 0.6]    10330
(0.6, 0.7]    12348
(0.7, 0.8]    17907
(0.8, 0.9]    36651
(0.9, 1.0]    23042
Name: count, dtype: int64


In [46]:

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

MAX_SEQ_LEN = model_phase1.cfg.max_seq_len
MAX_SOURCE_LEN = 768


class HybridEncoderSummarizationDataset(Dataset):

    def __init__(self, aligned_data):

        self.examples = []

        for doc in aligned_data:

            for chunk, target in zip(
                doc["chunks"],
                doc["chunk_targets"]
            ):

                if not target or not target.strip():
                    continue

                # Existing chunks already contain BOS and EOS.
                content_ids = list(chunk)

                if content_ids and content_ids[0] == BOS_ID:
                    content_ids = content_ids[1:]

                if content_ids and content_ids[-1] == EOS_ID:
                    content_ids = content_ids[:-1]

                # MAX_SOURCE_LEN includes BOS and EOS.
                content_ids = content_ids[:MAX_SOURCE_LEN - 2]

                source_prefix = (
                    [BOS_ID]
                    + content_ids
                    + [EOS_ID]
                )

                # Reserve one position for the final EOS.
                target_budget = (
                    MAX_SEQ_LEN - len(source_prefix) - 1
                )

                if target_budget <= 0:
                    continue

                target_ids = converter.to_tokens([target])[0]

                if not target_ids:
                    continue

                # Split long targets into segments.
                for start in range(
                    0,
                    len(target_ids),
                    target_budget
                ):

                    target_part = target_ids[
                        start:start + target_budget
                    ]

                    is_last = (
                        start + target_budget >= len(target_ids)
                    )

                    input_ids = source_prefix + target_part

                    labels = (
                        [-100] * len(source_prefix)
                        + target_part
                    )

                    if is_last:
                        input_ids.append(EOS_ID)
                        labels.append(EOS_ID)

                    self.examples.append({
                        "input_ids": input_ids,
                        "labels": labels
                    })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def hybrid_collate_fn(batch):

    input_ids = pad_sequence(
        [
            torch.tensor(x["input_ids"], dtype=torch.long)
            for x in batch
        ],
        batch_first=True,
        padding_value=PAD_ID
    )

    labels = pad_sequence(
        [
            torch.tensor(x["labels"], dtype=torch.long)
            for x in batch
        ],
        batch_first=True,
        padding_value=-100
    )

    padding_mask = input_ids != PAD_ID

    return {
        "input_ids": input_ids,
        "labels": labels,
        "padding_mask": padding_mask
    }


In [47]:
BATCH_SIZE = 32
from sklearn.model_selection import train_test_split

train_aligned, val_aligned = train_test_split(
    aligned_train_data,
    test_size=0.10,
    random_state=42,
    shuffle=True
)

print("Training documents:", len(train_aligned))
print("Validation documents:", len(val_aligned))

Training documents: 3677
Validation documents: 409


In [48]:
train_dataset = HybridEncoderSummarizationDataset(train_aligned)
val_dataset = HybridEncoderSummarizationDataset(val_aligned)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=hybrid_collate_fn,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=hybrid_collate_fn,
    pin_memory=True
)

print("Train examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Train examples: 24569
Validation examples: 2813


In [49]:
lorma_plus_model = LoRMAEncoder(
    base_model=model_phase1,
    rank=16,
    alpha=32,
    lorma_dropout=0.05,
    variant="plus"
).to(DEVICE)

lorma_plus_model.print_trainable_parameters()

Trainable parameters: 655,360
Total parameters:     57,247,744
Trainable percentage: 1.1448%


In [50]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape)

print("\nInput:")
print(tokenizer.decode(batch["input_ids"][0].tolist()))

print("\nLabels:")
valid_labels = batch["labels"][0]
valid_labels = valid_labels[valid_labels != -100]

print(tokenizer.decode(valid_labels.tolist()))

input_ids torch.Size([32, 1023])
labels torch.Size([32, 1023])
padding_mask torch.Size([32, 1023])

Input:
<s>tter of the Contempt of Court proceedings relating to the printing, publishing and circulation of a pamphlet over the name of Hira Lal Dixit (General Secretary, Praja Socialist Party, Mainpur) entitled "HAMARA VAHAN VIBRAG" arising out of (Civil) Petition No. 379 of 1953. (Hira Lal Dixit vs The State of Uttar Pradesh). The Attorney General for India (P. A. Mehta, with him) to assist the Court. S.C. Issacs (R. Patnaik and section section Shukla, with him) for respondent No. I (Hira Lal Dixit). Mohan Lal Saxena and section section Shukla for respondent No. 2 (Kishore Dutt Paliwal). S.S. Shukla for respondent No. 3 (Printer, Sainik Press). 679 1954. October 1. The Judgment of the Court was delivered by DAS J. This Rule was issued by this Court on the 16th September, 1954, calling upon the respondents to appear and show cause why they should not be proceeded against for contempt of

In [51]:
def print_gpu_memory():

    if not torch.cuda.is_available():
        print("CUDA unavailable")
        return

    torch.cuda.synchronize()

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    free, total = torch.cuda.mem_get_info()

    print(f"GPU total:      {total / 1024**3:.2f} GB")
    print(f"GPU free:       {free / 1024**3:.2f} GB")
    print(f"GPU allocated:  {allocated:.2f} GB")
    print(f"GPU reserved:   {reserved:.2f} GB")

print_gpu_memory()

GPU total:      14.56 GB
GPU free:       13.43 GB
GPU allocated:  0.87 GB
GPU reserved:   1.03 GB


In [52]:
# # 1. Load original model checkpoint
# checkpoint_path = "/kaggle/input/datasets/saiveerendra341251/decoder-p6/decoder_lora_epoch_7.pt"

# checkpoint = torch.load(
#     checkpoint_path,
#     map_location=DEVICE,
#     weights_only=False
# )

# # 2. Load LoRA model weights
# state_dict = checkpoint["model_state_dict"]

# state_dict = {
#     k.replace("module.", "", 1) if k.startswith("module.") else k: v
#     for k, v in state_dict.items()
# }

# # If lora_model is not wrapped in DataParallel yet
# lora_model.load_state_dict(state_dict, strict=True)

# print("Loaded epoch:", checkpoint["epoch"])
# print("Training loss:", checkpoint["train_loss"])

# 3. Wrap model in DataParallel
if torch.cuda.device_count() >= 2:
    lorma_plus_model = nn.DataParallel(
        lorma_plus_model,
        device_ids=[0, 1]
    )

lorma_plus_model = lorma_plus_model.to(DEVICE)

base = (
    lorma_plus_model.module
    if isinstance(lorma_plus_model, nn.DataParallel)
    else lorma_plus_model
)

optimizer = torch.optim.AdamW(
    base.get_trainable_parameters(),
    lr=2e-4,
    weight_decay=0.01
)

# # 5. Restore optimizer state
# optimizer.load_state_dict(
#     checkpoint["optimizer_state_dict"]
# )

# print("Optimizer state restored!")

In [53]:

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    scaler=None,
    use_amp=True
):

    model.train()

    total_loss = 0.0
    total_tokens = 0

    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    progress = tqdm(loader, desc="Training")

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        padding_mask = batch["padding_mask"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp and torch.cuda.is_available()
        ):

            logits = model(
                idx=input_ids,
                padding_mask=padding_mask,
                is_causal=True
            )

            # Next-token prediction
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss_sum = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            valid_tokens = (shift_labels != -100).sum()

            loss = loss_sum / valid_tokens.clamp_min(1)

        if scaler is not None and scaler.is_enabled():

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()

        else:

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                max_norm=1.0
            )

            optimizer.step()

        total_loss += loss_sum.detach().item()
        total_tokens += valid_tokens.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return total_loss / max(total_tokens, 1)


In [54]:

@torch.no_grad()
def evaluate(
    model,
    loader,
    device,
    use_amp=True
):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    progress = tqdm(loader, desc="Validation")

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        padding_mask = batch["padding_mask"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp and torch.cuda.is_available()
        ):

            logits = model(
                idx=input_ids,
                padding_mask=padding_mask,
                is_causal=True
            )

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss_sum = F.cross_entropy(
                shift_logits.reshape(-1, shift_logits.size(-1)),
                shift_labels.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            valid_tokens = (shift_labels != -100).sum()

        total_loss += loss_sum.item()
        total_tokens += valid_tokens.item()

        progress.set_postfix(
            val_loss=f"{(
                loss_sum / valid_tokens.clamp_min(1)
            ).item():.4f}"
        )

    return total_loss / max(total_tokens, 1)


In [55]:
EPOCHS = 10

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=torch.cuda.is_available()
)
best_val_loss = float("inf")

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss = train_one_epoch(
        lorma_plus_model,
        train_loader,
        optimizer,
        DEVICE,
        scaler=scaler,
        use_amp=True
    )

    val_loss = evaluate(
        lorma_plus_model,
        val_loader,
        DEVICE,
        use_amp=True
    )

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": lorma_plus_model.module.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss
    }

    torch.save(
        checkpoint,
        f"lorma_encoder_epoch_{epoch+1}.pt"
    )

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save(
            checkpoint,
            "lorma_encoder_best.pt"
        )


Epoch 1/10


Training:   0%|          | 0/768 [00:00<?, ?it/s]

Validation:   0%|          | 0/88 [00:00<?, ?it/s]

Train Loss: 3.9611
Validation Loss: 3.7142

Epoch 2/10


Training:   0%|          | 0/768 [00:00<?, ?it/s]

Validation:   0%|          | 0/88 [00:00<?, ?it/s]

... (21 outputs left out)

Train Loss: 3.6513
Validation Loss: 3.4721

Epoch 10/10


Training:   0%|          | 0/768 [00:00<?, ?it/s]

Validation:   0%|          | 0/88 [00:00<?, ?it/s]

Train Loss: 3.6406
Validation Loss: 3.4628


In [56]:
import json

OUTPUT_PATH = "train_chunk_aligned.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        aligned_train_data,
        f,
        ensure_ascii=False
    )

print(f"Saved {len(aligned_train_data)} documents to {OUTPUT_PATH}")

Saved 4086 documents to train_chunk_aligned.json


In [57]:
sample = test_df.iloc[0]

document = sample["document"]
document_id = sample["id"]

print("Document ID:", document_id)
print("Document length:", len(converter.to_tokens([document])[0]))

Document ID: 4963
Document length: 1687


In [58]:
import numpy as np
import torch

PAD_ID = 0
BOS_ID = 2
EOS_ID = 3

MAX_SEQ_LEN = 1024
CHUNK_OVERLAP = 128
MAX_CHUNKS = 20

MAX_SOURCE_LEN = 768
CHUNK_OVERLAP = 128

def split_document_into_chunks(
    document,
    max_chunks=20
):
    token_ids = converter.to_tokens([document])[0]

    payload_len = MAX_SOURCE_LEN - 2
    stride = payload_len - CHUNK_OVERLAP

    chunks = []

    for start in range(0, len(token_ids), stride):

        content = token_ids[start:start + payload_len]

        if not content:
            break

        chunk = [BOS_ID] + content + [EOS_ID]
        chunks.append(chunk)

        if start + payload_len >= len(token_ids):
            break

    if len(chunks) > max_chunks:
        indices = np.linspace(
            0,
            len(chunks) - 1,
            max_chunks,
            dtype=int
        )
        chunks = [chunks[i] for i in indices]

    return chunks

In [59]:
def sample_next_token(
    logits,
    generated_ids,
    temperature=0.7,
    top_k=40,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3
):

    logits = logits.clone()

    # Repetition penalty
    if repetition_penalty != 1.0:

        for token_id in set(generated_ids):

            if logits[0, token_id] > 0:
                logits[0, token_id] /= repetition_penalty
            else:
                logits[0, token_id] *= repetition_penalty

    # No-repeat n-gram
    n = no_repeat_ngram_size

    if n > 0 and len(generated_ids) >= n - 1:

        prefix = generated_ids[-(n - 1):]

        for i in range(len(generated_ids) - n + 1):

            if generated_ids[i:i + n - 1] == prefix:

                banned_token = generated_ids[i + n - 1]
                logits[0, banned_token] = float("-inf")

    logits = logits / temperature

    if top_k > 0:

        k = min(top_k, logits.size(-1))

        values, _ = torch.topk(logits, k, dim=-1)

        threshold = values[:, -1].unsqueeze(-1)

        logits = logits.masked_fill(
            logits < threshold,
            float("-inf")
        )

    probabilities = F.softmax(logits, dim=-1)

    return torch.multinomial(probabilities, 1)

In [60]:
@torch.no_grad()
def generate_summary_hybrid(
    model,
    chunk,
    max_new_tokens=200,
    temperature=0.7,
    top_k=40,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    do_sample=True,
    device=DEVICE
):

    model.eval()

    # Unwrap DataParallel
    if isinstance(model, nn.DataParallel):
        model = model.module

    base_model = model.model

    # Source
    src = torch.tensor(
        [chunk],
        dtype=torch.long,
        device=device
    )

    source_valid_mask = src != PAD_ID

    # Encode source once
    encoder_outputs = base_model.encoder(
        src,
        padding_mask=source_valid_mask
    )

    # Decoder starts with BOS
    decoder_ids = [BOS_ID]

    max_target_len = base_model.decoder.cfg.max_seq_len

    for step in range(max_new_tokens):

        if len(decoder_ids) >= max_target_len:
            break

        tgt = torch.tensor(
            [decoder_ids],
            dtype=torch.long,
            device=device
        )

        target_valid_mask = tgt != PAD_ID

        # Full-prefix decoder forward
        logits = base_model.decoder(
            idx=tgt,
            encoder_inputs=encoder_outputs,
            target_padding_mask=target_valid_mask,
            source_padding_mask=source_valid_mask
        )

        next_logits = logits[:, -1, :]

        if do_sample:

            next_token = sample_next_token(
                logits=next_logits,
                generated_ids=decoder_ids,
                temperature=temperature,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size
            )

        else:

            next_token = torch.argmax(
                next_logits,
                dim=-1,
                keepdim=True
            )

        token_id = next_token.item()

        if token_id == EOS_ID:
            break

        decoder_ids.append(token_id)

    # Exclude BOS
    summary_ids = decoder_ids[1:]

    return tokenizer.decode(summary_ids)

In [61]:

@torch.no_grad()
def generate_summary_hybrid(
    model,
    chunk,
    max_new_tokens=200,
    temperature=0.7,
    top_k=40,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
    do_sample=True,
    device=DEVICE
):

    model.eval()

    # Unwrap DataParallel
    if isinstance(model, nn.DataParallel):
        model = model.module

    base_model = model.model

    # Remove existing BOS/EOS if present.
    content = list(chunk)

    if content and content[0] == BOS_ID:
        content = content[1:]

    if content and content[-1] == EOS_ID:
        content = content[:-1]

    max_seq_len = base_model.cfg.max_seq_len

    # Keep space for generated summary.
    max_source_content = max_seq_len - 2 - max_new_tokens
    content = content[:max_source_content]

    # Prompt: BOS + document + EOS
    generated_ids = (
        [BOS_ID] + content + [EOS_ID]
    )

    for _ in range(max_new_tokens):

        input_ids = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device
        )

        padding_mask = input_ids != PAD_ID

        logits = model(
            idx=input_ids,
            padding_mask=padding_mask,
            is_causal=True
        )

        next_logits = logits[:, -1, :]

        if do_sample:

            next_token = sample_next_token(
                logits=next_logits,
                generated_ids=generated_ids,
                temperature=temperature,
                top_k=top_k,
                repetition_penalty=repetition_penalty,
                no_repeat_ngram_size=no_repeat_ngram_size
            )

        else:

            next_token = torch.argmax(
                next_logits,
                dim=-1,
                keepdim=True
            )

        token_id = next_token.item()

        if token_id == EOS_ID:
            break

        generated_ids.append(token_id)

        if len(generated_ids) >= max_seq_len:
            break

    # Remove BOS, document, and source EOS.
    summary_start = len(content) + 2
    summary_ids = generated_ids[summary_start:]

    return tokenizer.decode(summary_ids)


In [62]:
def generate_document_summary_hybrid(
    model,
    document,
    max_chunks=20,
    max_new_tokens=200
):

    chunks = split_document_into_chunks(
        document,
        max_chunks=max_chunks
    )

    chunk_summaries = []

    print("Total chunks:", len(chunks))

    for i, chunk in enumerate(chunks):

        summary = generate_summary_hybrid(
            model=model,
            chunk=chunk,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_k=40,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3
        )

        chunk_summaries.append(summary)

        generated_tokens = len(
            converter.to_tokens([summary])[0]
        )

        print(
            f"Chunk {i + 1}/{len(chunks)} | "
            f"Generated tokens: {generated_tokens}"
        )

    return chunk_summaries

In [63]:
import json
from tqdm.auto import tqdm


def generate_all_predictions_hybrid(
    model,
    test_data,
    output_path="hybrid_lora_predictions.json",
    max_chunks=20,
    max_new_tokens=200
):

    predictions = []

    for item in tqdm(test_data, desc="Generating predictions"):

        doc_id = str(item["id"])
        document = item["document"]

        chunk_summaries = generate_document_summary_hybrid(
            model=model,
            document=document,
            max_chunks=max_chunks,
            max_new_tokens=max_new_tokens
        )

        final_summary = " ".join(
            summary.strip()
            for summary in chunk_summaries
            if summary.strip()
        )

        predictions.append({
            "id": doc_id,
            "summary": final_summary
        })

    with open(output_path, "w", encoding="utf-8") as f:

        json.dump(
            predictions,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\nGeneration completed!")
    print("Total predictions:", len(predictions))
    print("Saved to:", output_path)

    return predictions

In [64]:
predictions = generate_all_predictions_hybrid(
    model=lorma_plus_model,
    test_data=test_data,
    max_chunks=20,
    max_new_tokens=200
)

Generating predictions:   0%|          | 0/100 [00:00<?, ?it/s]

Total chunks: 3
Chunk 1/3 | Generated tokens: 127
Chunk 2/3 | Generated tokens: 78
Chunk 3/3 | Generated tokens: 128
Total chunks: 12
Chunk 1/12 | Generated tokens: 74
Chunk 2/12 | Generated tokens: 40
Chunk 3/12 | Generated tokens: 200
Chunk 4/12 | Generated tokens: 110
Chunk 5/12 | Generated tokens: 60
Chunk 6/12 | Generated tokens: 57
Chunk 7/12 | Generated tokens: 115
Chunk 8/12 | Generated tokens: 175
Chunk 9/12 | Generated tokens: 49
Chunk 10/12 | Generated tokens: 127
Chunk 11/12 | Generated tokens: 30
Chunk 12/12 | Generated tokens: 200
Total chunks: 12
Chunk 1/12 | Generated tokens: 103
Chunk 2/12 | Generated tokens: 142
Chunk 3/12 | Generated tokens: 88
Chunk 4/12 | Generated tokens: 131
Chunk 5/12 | Generated tokens: 200
Chunk 6/12 | Generated tokens: 200
Chunk 7/12 | Generated tokens: 175
Chunk 8/12 | Generated tokens: 107
Chunk 9/12 | Generated tokens: 45
Chunk 10/12 | Generated tokens: 43
Chunk 11/12 | Generated tokens: 200
Chunk 12/12 | Generated tokens: 172
Total chunks